# FinMoE — demo

Short end-to-end demo for Kaggle (Accelerator: **GPU T4**, Internet: **on**, about 15 minutes):
the frozen model on a few financial sentences, one ESFT-Token training run, the same sentences
after adaptation, and the result tables of the full experiment grid.

In [ ]:
import os, subprocess
if not os.path.exists("/kaggle/working/Moe"):
    subprocess.run(["git", "clone", "-q", "https://github.com/2Gpb/Moe.git", "/kaggle/working/Moe"], check=True)
os.chdir("/kaggle/working/Moe")
!pip install -q uv
!uv pip install --system -q -e .
!python -m pytest -q tests

## 1. The frozen model on financial text

In [ ]:
SENTENCES = [
    "Operating loss narrowed to EUR 2.1 mn from EUR 5.4 mn a year earlier.",
    "The company will publish its interim report on 5 May.",
    "Net sales decreased by 12% due to weaker demand in Europe.",
    "Shares jumped 8% after the quarterly results beat expectations.",
]

def classify(run=None):
    for s in SENTENCES:
        cmd = ["python", "scripts/predict.py", "--task", "fpb", s] + (["--run", run] if run else [])
        out = subprocess.run(cmd, capture_output=True, text=True).stdout.strip().splitlines()
        print(s)
        print("   ", " | ".join(line.strip() for line in out))

classify()

## 2. Train ESFT-Token (82 experts, one epoch, about 7 minutes on a T4)

In [ ]:
!cat configs/esft_token.yaml
!python scripts/train.py --config configs/esft_token.yaml --seed 0 --out runs 2>&1 | grep -v Loading

## 3. The same sentences after adaptation

In [ ]:
classify("runs/esft_token_s0")

## 4. Results of the full grid (7 methods x 3 seeds)

In [ ]:
from IPython.display import Markdown, display
display(Markdown(open("results/main_table.md").read()))
display(Markdown(open("results/efficiency_table.md").read()))